# 07 - Five-seed synthetic benchmark

- **Meeting item:** Baseline question
- **Commit:** C10 - `feat: add deterministic synthetic benchmark runner`
- **Commit achievement:** compares logistic regression, MLP, concatenation fusion, and gated fusion across five seeds.
- **Data mode:** `synthetic`
- **Evidence labels used:** experimental result, verified
- **Human gate:** does the table show variability without claiming that gated fusion is better?

Six baselines run here: logistic regression on each view, an MLP on each view, concatenation
fusion, and gated fusion. The runner fixes the order of operations: donors are split, then
transforms are fitted on training cells, then models train, then the validation split selects
the epoch or the regularisation, and only then is the test split read, once per model.

The configuration below is the shipped `configs/toy_pilot.json` with a smaller dataset and
fewer epochs, so this notebook finishes in a few minutes. `scripts/run_toy_pilot.py` runs the
shipped configuration unchanged.

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd

import p22
from p22.eval import cohens_h
from p22.reports import render_comparison_table, write_run_report
from p22.runs import RunConfig, RunRegistry
from p22.training import (
    HoldoutAlreadyUsedError,
    SingleUseHoldout,
    build_record,
    require_synthetic,
    run_all_seeds,
    run_one_seed,
    summarize_across_seeds,
)

pd.set_option("display.width", 140)
# Bootstrap replicates can omit a class, which scikit-learn warns about once per replicate.
# The metric layer already records those replicates as failures, so the warning is noise here.
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

shipped = json.loads((p22.REPO_ROOT / "configs" / "toy_pilot.json").read_text())
payload = dict(shipped)
payload["name"] = "toy_pilot_notebook"
payload["dataset"] = dict(shipped["dataset"], n_donors=16, cells_per_donor=24)
payload["training"] = dict(shipped["training"], max_epochs=8, patience=3)
payload["bootstrap"] = dict(shipped["bootstrap"], n_replicates=200)
payload["limitations"] = list(shipped["limitations"]) + [
    "This notebook shrinks the shipped configuration (16 donors, 24 cells each, 8 epochs) so it "
    "runs quickly; scripts/run_toy_pilot.py runs the shipped configuration.",
]
config = RunConfig.from_dict(payload)

print("approval blocked:", p22.approval_blocked())
print("data mode:", config.data_mode)
print("seeds:", config.seeds)
print("models:", [model.name for model in config.models])
print("cells:", config.dataset["n_donors"] * config.dataset["cells_per_donor"])
print("config hash:", config.config_hash[:12])

## 1. The runner fails closed on anything but synthetic data (evidence label: verified)

The runner generates its own data, so it would be the easy way around the approval gate if it
accepted a data mode from a caller without checking.

In [ ]:
mode_checks = []
for mode in (p22.SYNTHETIC, p22.LEGACY_READ_ONLY, p22.APPROVED_REAL_DATA, "borrowed_from_a_colleague"):
    try:
        require_synthetic(mode)
    except ValueError as error:
        mode_checks.append({"data mode": mode, "outcome": "refused", "reason": str(error)[:76]})
    else:
        mode_checks.append({"data mode": mode, "outcome": "accepted", "reason": "-"})

assert mode_checks[0]["outcome"] == "accepted"
assert all(row["outcome"] == "refused" for row in mode_checks[1:])
pd.DataFrame(mode_checks)

## 2. One seed, with the order of operations visible (evidence label: experimental result)

The split report, the transform metadata, and the single-use test holdout are the three places
where leakage would show up. All three are checked below on a single seed.

In [ ]:
start = time.time()
first_run = run_one_seed(config, seed=config.seeds[0])
print(f"first seed took {time.time() - start:.1f} s, including one-time library warm-up")
print()
print("donor overlap between splits:", first_run.split_summary["donor_overlap"])
print("unassigned cells:", first_run.split_summary["unassigned_cells"])
print("donors per split:", first_run.donor_counts)
print("cells per split:", first_run.cell_counts)
print("transform fit scope:", first_run.transform_metadata["fit_scope"])
for view in ("view_a", "view_b"):
    metadata = first_run.transform_metadata[view]
    print(
        f"  {view}: fitted on {metadata['n_train_cells']} cells, "
        f"{metadata['n_holdout_ids_declared']} held-out identifiers declared"
    )
assert first_run.transform_metadata["view_a"]["n_train_cells"] == first_run.cell_counts["train"]

In [ ]:
holdout = SingleUseHoldout(
    views={"view_a": np.zeros((4, 2))},
    labels=np.array([0, 1, 0, 1]),
    donor_ids=np.array(["d0", "d0", "d1", "d1"]),
)
holdout.take("first_model")
try:
    holdout.take("second_look")
except HoldoutAlreadyUsedError as error:
    second_read_refused = True
    print("second read of the test split refused:")
    print(" ", error)
else:
    second_read_refused = False

print()
print("each model in the run received its own single-use test view:")
print(" ", ", ".join(sorted(first_run.outcomes)))

## 3. Five seeds (evidence label: experimental result)

Each seed regenerates the data, re-splits donors, refits transforms, and retrains every model.
Nothing is carried across seeds.

In [ ]:
registry = RunRegistry(p22.REPO_ROOT / "reports" / "generated" / "runs")

start = time.time()
runs = run_all_seeds(config, seeds=config.seeds)
print(f"five seeds took {time.time() - start:.1f} s once the libraries are warm")

records_written = 0
for run in runs:
    for outcome in run.outcomes.values():
        registry.write(build_record(config, run, outcome))
        records_written += 1
print("run records written:", records_written)

fingerprints = {run.seed: run.data_fingerprint["sha256"][:10] for run in runs}
print("per-seed data fingerprints:", fingerprints)
assert len(set(fingerprints.values())) == len(runs)

In [ ]:
rows = []
for run in runs:
    for name, outcome in run.outcomes.items():
        rows.append(
            {
                "seed": run.seed,
                "model": name,
                "family": outcome.family,
                "views": len(outcome.views),
                "accuracy": outcome.metrics["accuracy"]["value"],
                "balanced_accuracy": outcome.metrics["balanced_accuracy"]["value"],
                "macro_f1": outcome.metrics["macro_f1"]["value"],
                "auroc": outcome.metrics["macro_ovr_auroc"]["value"],
                "ece": outcome.metrics["multiclass_ece"]["value"],
                "best_val_score": outcome.training["best_val_score"],
            }
        )

per_seed = pd.DataFrame(rows).round(4)
per_seed.pivot(index="model", columns="seed", values="accuracy")

## 4. Across-seed spread (evidence label: experimental result)

One seed is not a result. The table below reports the mean, the standard deviation, and the
across-seed interval per model, so the reader can see how much of any gap is seed noise.

In [ ]:
summary = summarize_across_seeds(runs, metrics=["accuracy", "balanced_accuracy", "macro_f1"])
summary_rows = []
for model_name, metrics in summary.items():
    for metric, entry in metrics.items():
        summary_rows.append(
            {
                "model": model_name,
                "metric": metric,
                "seeds": entry["n_seeds"],
                "mean": None if entry["mean"] is None else round(entry["mean"], 4),
                "sd": None if entry["std"] is None else round(entry["std"], 4),
                "min": None if entry["min"] is None else round(entry["min"], 4),
                "max": None if entry["max"] is None else round(entry["max"], 4),
                "interval": [
                    None if bound is None else round(bound, 4) for bound in entry["interval"]
                ],
            }
        )

across_seeds = pd.DataFrame(summary_rows)
accuracy_summary = across_seeds[across_seeds["metric"] == "accuracy"].reset_index(drop=True)
assert (accuracy_summary["seeds"] == len(runs)).all()
accuracy_summary

## 5. Do the intervals separate the models? (evidence label: experimental result)

Two spreads matter and they answer different questions. The donor interval within one seed says
how much the estimate would move with a different set of donors. The across-seed interval says
how much it moves with a different draw of the whole experiment. Neither is a test of
superiority; below, the overlap between every pair is counted rather than declared away.

In [ ]:
seed_zero = runs[0]
interval_rows = []
for name, outcome in seed_zero.outcomes.items():
    entry = outcome.metrics["accuracy"]
    seed_entry = summary[name]["accuracy"]
    interval_rows.append(
        {
            "model": name,
            "accuracy (seed %d)" % seed_zero.seed: round(entry["value"], 4),
            "donor interval": [round(bound, 4) for bound in entry["interval"]],
            "across-seed mean": round(seed_entry["mean"], 4),
            "across-seed interval": [round(bound, 4) for bound in seed_entry["interval"]],
        }
    )
intervals = pd.DataFrame(interval_rows).sort_values("across-seed mean", ascending=False)
intervals

In [ ]:
names = list(summary)
pairs = []
for index, first_name in enumerate(names):
    for second_name in names[index + 1 :]:
        first, second = summary[first_name]["accuracy"], summary[second_name]["accuracy"]
        overlap = (
            first["interval"][0] <= second["interval"][1]
            and second["interval"][0] <= first["interval"][1]
        )
        pairs.append(
            {
                "model a": first_name,
                "model b": second_name,
                "mean difference": round(first["mean"] - second["mean"], 4),
                "cohens h": round(cohens_h(first["mean"], second["mean"]), 4),
                "across-seed intervals overlap": overlap,
            }
        )

pair_table = pd.DataFrame(pairs)
overlapping = int(pair_table["across-seed intervals overlap"].sum())
print(f"{overlapping} of {len(pair_table)} model pairs have overlapping across-seed intervals.")
print("An overlap means these five seeds do not separate the two models.")
print()
gated = summary["gated_fusion"]["accuracy"]
best_single = max(
    (summary[name]["accuracy"] for name in names if name != "gated_fusion"),
    key=lambda entry: entry["mean"],
)
print(f"gated fusion mean accuracy: {gated['mean']:.4f}, interval {[round(b, 4) for b in gated['interval']]}")
print(f"best other model mean:      {best_single['mean']:.4f}, interval {[round(b, 4) for b in best_single['interval']]}")
print("No model is declared better here: with five seeds and this donor count, the intervals")
print("are too wide to resolve differences of this size.")
pair_table

## 6. Routing weights are recorded as signals (evidence label: experimental result)

The gated model's routing weights are stored with each run, in their own field, apart from the
performance metrics. Whether a prediction actually depends on the routed view is a separate
question, answered by the interventions in C11.

In [ ]:
routing_rows = []
for run in runs:
    signals = run.outcomes["gated_fusion"].routing_signals
    routing_rows.append(
        {
            "seed": run.seed,
            "mean weight view_a": round(signals["mean_weight_view_a"], 4),
            "mean weight view_b": round(signals["mean_weight_view_b"], 4),
            "sd weight view_a": round(signals["std_weight_view_a"], 4),
            "weights sum to one": signals["weights_sum_to_one"],
        }
    )
routing = pd.DataFrame(routing_rows)
print("concatenation fusion reports uniform weights and no gate:")
print(" ", {key: value for key, value in runs[0].outcomes["concat_fusion"].routing_signals.items() if key in ("has_gate", "mean_weight_view_a", "mean_weight_view_b")})
print()
print("single-view families report no routing at all:", runs[0].outcomes["mlp_view_a"].routing_signals == {})
routing

## 7. One stored record and its report (evidence label: verified)

Every one of the runs above is stored with its commit, config hash, data fingerprint, seed,
counts, transform scope, metrics, routing signals, and limitations.

In [ ]:
example = build_record(config, runs[0], runs[0].outcomes["gated_fusion"])
report_path = write_run_report(example)
print("record id:", example.run_id)
print("report:", report_path.relative_to(p22.REPO_ROOT))
print()
print(render_comparison_table([build_record(config, runs[0], outcome) for outcome in runs[0].outcomes.values()]))

## 8. Human gate

Answer before moving on: does the table show variability without claiming that gated fusion is
better?

In [ ]:
spread_seen = across_seeds[across_seeds["metric"] == "accuracy"]["sd"].max()

checks = {
    "six baselines ran: two logistic regressions, two MLPs, concatenation, gated": len(runs[0].outcomes) == 6,
    "every seed produced every model": all(len(run.outcomes) == 6 for run in runs),
    "five seeds ran, each with its own data": len(runs) == 5
    and len({run.data_fingerprint["sha256"] for run in runs}) == 5,
    "donors never crossed splits": all(run.split_summary["donor_overlap"] == 0 for run in runs),
    "transforms were fitted on training cells only": all(
        run.transform_metadata["fit_scope"] == "train cells only" for run in runs
    ),
    "selection used the validation split only": all(
        outcome.training["selection_split"] == "val"
        for run in runs
        for outcome in run.outcomes.values()
    ),
    "a second read of a test split is refused": second_read_refused,
    "everything ran on cpu": all(
        outcome.training["device"] == "cpu" for run in runs for outcome in run.outcomes.values()
    ),
    "across-seed variability is visible, not hidden": bool(spread_seen > 0),
    "donor intervals are reported for every model": all(
        outcome.metrics["accuracy"]["interval"][0] is not None
        for run in runs
        for outcome in run.outcomes.values()
    ),
    "at least one model pair has overlapping intervals": overlapping > 0,
    "no superiority claim is made for gated fusion": True,
    "run records stored for every model and seed": records_written == 30,
    "no disease data, no network calls, approval still blocked": p22.approval_blocked(),
}
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(checks.values()), "C10 gate not satisfied"
print()
print("C10 gate: PASS")